# Genetic Algorithms: From Line Fitting to the Travelling Salesman

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/optimisation/genetic_algorithms.ipynb)

This notebook accompanies the blog post at [sesen.ai](https://sesen.ai/blog/genetic-algorithms-from-scratch).

We implement genetic algorithms from scratch for two problems:
1. **Line fitting** — evolve (a, b) coefficients for y = ax + b
2. **Travelling Salesman Problem** — evolve shortest route through cities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## Part 1: Evolving a Line

We generate noisy data from y = 3x + 8 and evolve (a, b) to fit it.

In [ ]:
def ga_line_fit(x, y, pop_size=100, n_elites=20, mutation_rate=0.1,
                generations=50):
    """Evolve (a, b) to minimise RMSE for y = ax + b."""
    pop = np.random.uniform(-10, 10, size=(pop_size, 2))
    history = []
    all_pops = []

    for gen in range(generations):
        # FITNESS: how well each individual fits the data
        preds = pop[:, 0:1] * x + pop[:, 1:2]
        rmse = np.sqrt(((preds - y) ** 2).mean(axis=1))
        fitness = 1.0 / (rmse + 1e-10)

        best_idx = np.argmax(fitness)
        history.append((pop[best_idx, 0], pop[best_idx, 1], rmse[best_idx]))
        all_pops.append(pop.copy())

        # SELECTION: keep elite performers + sample by fitness
        ranked = np.argsort(-fitness)
        elites = pop[ranked[:n_elites]]
        probs = fitness / fitness.sum()
        parents = pop[np.random.choice(pop_size, pop_size - n_elites, p=probs)]

        # CROSSOVER + MUTATION: average parents, add Gaussian noise
        children = np.empty_like(parents)
        for i in range(len(parents)):
            child = (parents[i] + parents[-(i + 1)]) / 2
            child *= 1 + mutation_rate * np.random.randn(2)
            children[i] = child

        pop = np.vstack([elites, children])

    return history, all_pops

In [ ]:
np.random.seed(42)
x_data = np.random.rand(50)
y_data = 3 * x_data + 8 + 0.3 * np.random.randn(50)

history, all_pops = ga_line_fit(x_data, y_data)
a, b, rmse = history[-1]
print(f"Evolved: y = {a:.2f}x + {b:.2f}  (RMSE: {rmse:.4f})")
print(f"True:    y = 3.00x + 8.00")

### Animate the Evolution

In [ ]:
frame_indices = [0, 1, 2, 3, 5, 8, 12, 20, 35, 49]

fig, ax = plt.subplots(figsize=(8, 5))
x_line = np.linspace(0, 1, 100)

def update_ols(frame_num):
    ax.clear()
    gen = frame_indices[frame_num]
    pop = all_pops[gen]
    a_best, b_best, rmse_best = history[gen]
    
    ax.scatter(x_data, y_data, c='gray', alpha=0.5, s=20)
    # Top 5 individuals
    preds = pop[:, 0:1] * x_data + pop[:, 1:2]
    rmses = np.sqrt(((preds - y_data) ** 2).mean(axis=1))
    top5 = np.argsort(rmses)[:5]
    for idx in top5:
        ax.plot(x_line, pop[idx, 0] * x_line + pop[idx, 1], 'b-', alpha=0.15, linewidth=1)
    ax.plot(x_line, a_best * x_line + b_best, 'b-', linewidth=2.5,
            label=f'Best: y = {a_best:.2f}x + {b_best:.2f}')
    ax.plot(x_line, 3 * x_line + 8, 'r--', alpha=0.6, linewidth=1.5, label='True: y = 3x + 8')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(4, 14)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Generation {gen}  |  RMSE: {rmse_best:.3f}')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)

anim = FuncAnimation(fig, update_ols, frames=len(frame_indices), interval=500)
HTML(anim.to_jshtml())

### Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([h[2] for h in history], 'b-o', markersize=4)
ax.set_xlabel('Generation')
ax.set_ylabel('RMSE')
ax.set_title('GA Convergence: Line Fitting')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.show()

## Part 2: The Travelling Salesman Problem

Now we tackle a combinatorial problem where gradients don't exist.

In [ ]:
def ordered_crossover(p1, p2):
    """Take a slice from parent 1, fill gaps from parent 2's order."""
    n = len(p1)
    start, end = sorted(np.random.choice(n, 2, replace=False))
    child = -np.ones(n, dtype=int)
    child[start:end] = p1[start:end]
    fill = [g for g in p2 if g not in child[start:end]]
    child[child == -1] = fill
    return child


def swap_mutate(route, rate):
    """Randomly swap cities with given probability per position."""
    route = route.copy()
    for i in range(len(route)):
        if np.random.random() < rate:
            j = np.random.randint(len(route))
            route[i], route[j] = route[j], route[i]
    return route


def ga_tsp(cities, pop_size=100, n_elites=20, mutation_rate=0.01,
           generations=500):
    """Evolve shortest route through all cities."""
    n = len(cities)

    def total_distance(route):
        shifts = np.roll(route, -1)
        return np.sqrt(((cities[route] - cities[shifts]) ** 2).sum(axis=1)).sum()

    pop = [np.random.permutation(n) for _ in range(pop_size)]
    history = []

    for gen in range(generations):
        distances = np.array([total_distance(r) for r in pop])
        fitness = 1.0 / distances
        history.append((pop[np.argmin(distances)].copy(), distances.min()))

        # Selection: elites + fitness-proportionate
        ranked = np.argsort(-fitness)
        elites = [pop[i].copy() for i in ranked[:n_elites]]
        probs = fitness / fitness.sum()
        idx = np.random.choice(pop_size, pop_size - n_elites, p=probs)
        np.random.shuffle(idx)

        # Crossover + mutation
        children = []
        for i in range(0, len(idx) - 1, 2):
            for p, q in [(idx[i], idx[i+1]), (idx[i+1], idx[i])]:
                children.append(swap_mutate(
                    ordered_crossover(pop[p], pop[q]), mutation_rate))
        children = children[:pop_size - n_elites]
        pop = elites + children

    return history

In [ ]:
np.random.seed(42)
cities = np.random.rand(25, 2) * 200

tsp_history = ga_tsp(cities)
best_route, best_dist = tsp_history[-1]
print(f"Initial distance: {tsp_history[0][1]:.0f}")
print(f"Evolved distance: {best_dist:.0f}")

### Animate the Route Evolution

In [ ]:
tsp_frame_indices = [0, 1, 3, 7, 15, 30, 60, 120, 250, 400, 499]

fig, ax = plt.subplots(figsize=(8, 8))

def update_tsp(frame_num):
    ax.clear()
    idx = tsp_frame_indices[frame_num]
    route, dist = tsp_history[idx]
    closed_route = np.append(route, route[0])
    ax.plot(cities[closed_route, 0], cities[closed_route, 1], 'b-', linewidth=1.5, alpha=0.7)
    ax.scatter(cities[:, 0], cities[:, 1], c='red', s=80, zorder=5, edgecolors='darkred')
    for i in range(len(cities)):
        ax.annotate(str(i), (cities[i, 0]+3, cities[i, 1]+3), fontsize=7, alpha=0.6)
    ax.set_xlim(-10, 210)
    ax.set_ylim(-10, 210)
    ax.set_title(f'Generation {idx}  |  Distance: {dist:.0f}', fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

anim = FuncAnimation(fig, update_tsp, frames=len(tsp_frame_indices), interval=500)
HTML(anim.to_jshtml())

### TSP Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([h[1] for h in tsp_history], 'b-', linewidth=1)
ax.set_xlabel('Generation')
ax.set_ylabel('Best Distance')
ax.set_title('GA Convergence: TSP (25 cities)')
ax.grid(True, alpha=0.3)
plt.show()

## GA vs Gradient Descent

For differentiable problems, gradient descent is much more efficient.

In [ ]:
def gd_line_fit(x, y, lr=0.1, iterations=50):
    """Gradient descent to fit y = ax + b."""
    a, b = 0.0, 0.0
    gd_history = []
    for _ in range(iterations):
        preds = a * x + b
        error = preds - y
        a -= lr * 2 * (error * x).mean()
        b -= lr * 2 * error.mean()
        gd_history.append((a, b, np.sqrt((error ** 2).mean())))
    return gd_history

np.random.seed(42)
x_data = np.random.rand(50)
y_data = 3 * x_data + 8 + 0.3 * np.random.randn(50)

ga_hist, _ = ga_line_fit(x_data, y_data)
gd_hist = gd_line_fit(x_data, y_data, lr=0.5, iterations=50)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([h[2] for h in ga_hist], 'b-o', label='Genetic Algorithm (pop=100)', markersize=4)
ax.plot([h[2] for h in gd_hist], 'r--s', label='Gradient Descent (lr=0.5)', markersize=4)
ax.set_xlabel('Generation / Iteration')
ax.set_ylabel('RMSE')
ax.set_yscale('log')
ax.set_title('GA vs Gradient Descent: Line Fitting')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"GA final:  a={ga_hist[-1][0]:.4f}, b={ga_hist[-1][1]:.4f}, RMSE={ga_hist[-1][2]:.4f}")
print(f"GD final:  a={gd_hist[-1][0]:.4f}, b={gd_hist[-1][1]:.4f}, RMSE={gd_hist[-1][2]:.4f}")

## Exercises

1. **Noise sensitivity** — Add increasing noise (std=0.5, 1.0, 2.0) to the line data. How does the GA's RMSE floor change?
2. **More cities** — Try 50 or 100 cities in the TSP. How does convergence change? Do you need more generations?
3. **Tournament selection** — Replace roulette wheel with tournament selection: pick k random individuals, keep the best. Compare convergence speed.
4. **GA for neural nets** — Use GA to evolve the weights of a 2-2-1 neural network for XOR (see the [backpropagation notebook](../deep-learning/backpropagation.ipynb)). How many generations does it take?
5. **Mutation rate sweep** — Run the TSP GA with mutation rates [0.001, 0.01, 0.05, 0.1, 0.2]. Plot convergence curves for each. Where is the sweet spot?